# Model Evaluation
## Explainable Fake News Detection using Transformer-based Language Models

**Student Name:** Abdullah  
**Reg Number:** M24F0044DS009  
**Course:** Natural Language Processing  

### Objective
The objective of this notebook is to evaluate the performance of the fine-tuned BERT model on unseen test data using standard classification metrics.


In [ ]:
# Import Required Libraries

import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from transformers import BertTokenizer, BertForSequenceClassification
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import warnings
warnings.filterwarnings("ignore")


%matplotlib inline


In [ ]:
# Check Device Gpu or Cpu

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


In [ ]:
# Load Test Dataset
test_df = pd.read_csv("../data/shorttextpreprocessedtest.csv")
test_df.head()


In [ ]:
# Load Trained Model & Tokenizer

model_path = "../models/bert_fake_news_model"

tokenizer = BertTokenizer.from_pretrained(model_path)
model = BertForSequenceClassification.from_pretrained(model_path)
model.to(device)
model.eval()


In [ ]:
# Clean test text column (REQUIRED)
test_df = test_df[test_df['text'].notna()]
test_df['text'] = test_df['text'].astype(str)

# Tokenize Test Data
encodings = tokenizer(
    test_df['text'].tolist(),
    truncation=True,
    padding=True,
    max_length=128,
    return_tensors="pt"
)

input_ids = encodings['input_ids'].to(device)
attention_mask = encodings['attention_mask'].to(device)
labels = torch.tensor(test_df['label'].values).to(device)


In [ ]:
# Generate Predictions

with torch.no_grad():
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )
    
predictions = torch.argmax(outputs.logits, dim=1)


In [ ]:
# Convert Predictions to CPU

y_true = labels.cpu().numpy()
y_pred = predictions.cpu().numpy()


In [ ]:
# Compute Evaluation Metrics

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1-score :", f1)


####  Why these metrics?

- Accuracy → overall performance

- Precision → false positives control

- Recall → false negatives control

- F1-score → balance (important for fake news)

In [ ]:
# Detailed Classification Report

print(classification_report(
    y_true,
    y_pred,
    target_names=["Fake News", "Real News"]
))


In [ ]:
# Confusion Matrix

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6,4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Fake", "Real"],
    yticklabels=["Fake", "Real"]
)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.show()


- Diagonal = correct predictions

- Off-diagonal = misclassifications

In [ ]:
# Save Metrics to File

with open("../results/metrics.txt", "w") as f:
    f.write(f"Accuracy: {accuracy}\n")
    f.write(f"Precision: {precision}\n")
    f.write(f"Recall: {recall}\n")
    f.write(f"F1-score: {f1}\n")


## Evaluation Summary

- The fine-tuned BERT model demonstrates strong performance on unseen test data.
- High F1-score indicates balanced classification of fake and real news.
- The confusion matrix shows effective discrimination with minimal misclassification.
- These results validate the effectiveness of Transformer-based models for fake news detection.
